# GLM-4.6V AWQ 4bit — Zero-shot VQA

Qwen3.5-122B 검증 노트북과 같은 목적의 **깡모델 zero-shot** 실행본입니다.

- 모델: `cyankiwi/GLM-4.6V-AWQ-4bit`
- 런타임: **A100 80GB 또는 G4 96GB**
- 학습/LoRA 없음
- OCR/DINO 없음
- 외부 API 없음
- 실제 이미지 + 질문 + a/b/c/d
- structured output으로 `a/b/c/d`만 허용
- Test 6,714개 resume + submission 생성

> 이전 LoRA 실험에서 `Transformers`가 compressed experts를 풀면서 OOM이 났던 경로를 피하고,
> 이번에는 **vLLM로 AWQ/compressed-tensors 양자화 모델을 그대로 추론**합니다.


## Step 0. GLM vLLM 환경 설치

이 셀 실행 후 **런타임을 한 번 재시작**하고 Step 1부터 진행하세요.


In [ ]:
# ============================================================
# STEP 0. INSTALL GLM-4.6V runtime
# 모델 카드 요구사항: vLLM >= 0.12 / Transformers >= 5.0 rc
# 완료 후 런타임 재시작 1회
# ============================================================

import sys, subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "vllm>=0.12.0",
    "transformers>=5.0.0rc0",
    "compressed-tensors",
    "huggingface_hub",
    "pillow",
    "pandas",
], check=True)

print("\n✅ 설치 완료")
print("이제 Colab: 런타임 > 세션 다시 시작")
print("재시작 후 STEP 1부터 실행하세요.")


## Step 1. GLM-4.6V AWQ 4bit 로드


In [ ]:
# ============================================================
# STEP 1. LOAD GLM-4.6V AWQ 4bit
# A100 80GB / G4 96GB
# ============================================================

import torch, vllm
from vllm import LLM

MODEL_ID = "cyankiwi/GLM-4.6V-AWQ-4bit"

assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 2**30

print("GPU :", props.name)
print("VRAM:", f"{vram_gib:.1f} GiB")
print("vLLM:", vllm.__version__)

if vram_gib < 78:
    raise RuntimeError("A100 80GB 또는 G4 96GB급 단일 GPU를 사용하세요.")

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    tensor_parallel_size=1,
    dtype="bfloat16",
    max_model_len=4096,
    max_num_seqs=1,
    gpu_memory_utilization=0.96,
    enforce_eager=True,
    limit_mm_per_prompt={"image": 1},
    seed=42,
)

print("✅ GLM model loaded:", MODEL_ID)


## Step 2. Test 데이터와 VQA 프롬프트 준비


In [ ]:
# ============================================================
# STEP 2. DATA + PROMPT
# - /content/drive/MyDrive/data.zip (비밀번호 없는 일반 ZIP)
# - test.csv / test/ / sample_submission.csv 자동 확인
# - OCR / DINO / 외부 API 없음
# ============================================================

from pathlib import Path
import zipfile, shutil, os, time, inspect
import pandas as pd
from PIL import Image, ImageOps
from google.colab import drive
from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams

drive.mount("/content/drive")

DATA_ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/data.zip"),
    Path("/content/drive/MyDrive/mori/data.zip"),
]
DATA_ZIP = next((p for p in DATA_ZIP_CANDIDATES if p.is_file()), None)
if DATA_ZIP is None:
    raise FileNotFoundError(f"data.zip을 찾지 못했습니다: {DATA_ZIP_CANDIDATES}")

DATA_ROOT = Path("/content/vqa_data_plain")

def prepare_plain_zip(zip_path, out_dir):
    out_dir = Path(out_dir)

    # 이미 준비된 경우 그대로 재사용
    roots = []
    if out_dir.exists():
        roots = sorted({
            p.parent for p in out_dir.rglob("test.csv")
            if (p.parent / "sample_submission.csv").is_file()
            and (p.parent / "test").is_dir()
        })
    if len(roots) == 1:
        print("기존 데이터 재사용:", roots[0])
        return roots[0]

    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("일반 ZIP 압축 해제:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        encrypted = [x.filename for x in zf.infolist() if x.flag_bits & 1]
        if encrypted:
            raise RuntimeError(f"암호화된 파일이 있습니다: {encrypted[:5]}")
        zf.extractall(out_dir)

    roots = sorted({
        p.parent for p in out_dir.rglob("test.csv")
        if (p.parent / "sample_submission.csv").is_file()
        and (p.parent / "test").is_dir()
    })
    if len(roots) != 1:
        raise RuntimeError(f"test 데이터 root를 하나 찾지 못했습니다: {roots}")
    return roots[0]

DATA_ROOT = prepare_plain_zip(DATA_ZIP, DATA_ROOT)
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

test_df = pd.read_csv(TEST_CSV)
sample_submission = pd.read_csv(SAMPLE_CSV, dtype={"id": str})

required = ["id","path","question","a","b","c","d"]
missing = [c for c in required if c not in test_df.columns]
assert not missing, f"test.csv 컬럼 누락: {missing}"
assert len(test_df) == len(sample_submission), (len(test_df), len(sample_submission))
assert test_df["id"].astype(str).tolist() == sample_submission["id"].astype(str).tolist()

exists = test_df["path"].map(lambda p: (DATA_ROOT / str(p)).is_file())
assert exists.all(), f"누락 이미지 {int((~exists).sum())}개"

SYSTEM_PROMPT = (
    "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
    "이미지를 가장 중요한 근거로 사용하고 질문과 선택지를 신중하게 비교하세요.\n"
    "최종 답은 반드시 a, b, c, d 중 소문자 한 글자로만 출력하세요."
)

def build_prompt(row):
    return (
        f"질문: {row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        "정답:"
    )

choice_constraint = StructuredOutputsParams(choice=["a","b","c","d"])
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=2,
    structured_outputs=choice_constraint,
)

SUPPORTS_USE_TQDM = "use_tqdm" in inspect.signature(llm.chat).parameters

def load_image(path, max_side=None):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        if max_side is not None:
            im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        return im.copy()

def infer_one(row, max_side=None):
    image_path = DATA_ROOT / str(row["path"])
    image = load_image(image_path, max_side=max_side)

    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":[
            {"type":"image_pil","image_pil":image},
            {"type":"text","text":build_prompt(row)},
        ]},
    ]

    kwargs = dict(
        messages=messages,
        sampling_params=sampling_params,
        chat_template_kwargs={"enable_thinking": False},
    )
    if SUPPORTS_USE_TQDM:
        kwargs["use_tqdm"] = False

    outputs = llm.chat(**kwargs)
    pred = outputs[0].outputs[0].text.strip().lower()

    if pred not in {"a","b","c","d"}:
        raise ValueError(f"비정상 prediction: {pred!r}")
    return pred

print("=" * 72)
print("DATA READY")
print("ROOT :", DATA_ROOT)
print("TEST :", len(test_df))
print("IMAGE:", int(exists.sum()))
print("=" * 72)

display(test_df.head(3))


## Step 3. 3문제 smoke


In [ ]:
# ============================================================
# STEP 3. 3개 TEST SMOKE
# - 정답 없음. 모델/이미지/프롬프트 경로가 정상인지 확인용.
# ============================================================

for i in range(3):
    row = test_df.iloc[i]
    t0 = time.time()
    pred = infer_one(row)
    print(
        f"{i}: {row['id']} -> {pred} "
        f"({time.time()-t0:.2f}s) | {row['question']}"
    )

print("✅ smoke 완료. 위 3개가 a/b/c/d로 나오면 STEP 4 실행.")


## Step 4. Test 6,714개 전체 추론 + 제출


In [ ]:
# ============================================================
# STEP 4. FULL TEST 6,714 + RESUME + SUBMISSION
# - 깡모델 zero-shot
# - 원본 이미지 우선
# - 실패한 문항만 1600 -> 1280 -> 1024로 자동 축소 재시도
# - 매 문제 Drive 저장
# - 같은 셀 재실행 시 완료된 ID 자동 skip
# ============================================================

import gc, os, time
from pathlib import Path
import pandas as pd
import torch

OUT_DIR = Path("/content/drive/MyDrive/cleaning_data/glm46v_awq4_eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = OUT_DIR / "glm46v_awq4_test_predictions.csv"
SUBMISSION_CSV = OUT_DIR / "glm46v_awq4_submission.csv"

results = {}

if RESULT_CSV.exists():
    old = pd.read_csv(RESULT_CSV, dtype={"id":str})
    for rec in old.drop_duplicates("id", keep="last").to_dict("records"):
        results[str(rec["id"])] = rec

done_ids = {
    k for k,v in results.items()
    if str(v.get("status","")) == "ok"
    and str(v.get("prediction","")).strip().lower() in {"a","b","c","d"}
}

def save_results():
    if not results:
        return
    x = pd.DataFrame(list(results.values())).sort_values("idx").reset_index(drop=True)
    tmp = Path(str(RESULT_CSV) + ".tmp")
    x.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, RESULT_CSV)

def hms(sec):
    sec = int(sec)
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

print("=" * 78)
print("GLM46V_AWQ4 ZERO-SHOT FULL TEST")
print("RESUME:", len(done_ids), "/", len(test_df))
print("RESULT:", RESULT_CSV)
print("=" * 78)

session_start = time.time()
session_done = 0
session_fail = 0

for idx, row in test_df.iterrows():
    img_id = str(row["id"])
    if img_id in done_ids:
        continue

    pred = ""
    status = "error"
    error = ""
    used_max_side = "original"
    t0 = time.time()

    # OCR가 중요한 대회이므로 원본을 먼저 사용.
    # context/vision token OOM·길이 문제만 자동 축소 재시도.
    retry_sizes = [None, 1600, 1280, 1024]

    for max_side in retry_sizes:
        try:
            pred = infer_one(row, max_side=max_side)
            status = "ok"
            error = ""
            used_max_side = "original" if max_side is None else str(max_side)
            break
        except Exception as e:
            error = repr(e)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    elapsed = time.time() - t0

    results[img_id] = {
        "idx": int(idx),
        "id": img_id,
        "path": row["path"],
        "prediction": pred,
        "status": status,
        "elapsed_sec": round(elapsed, 4),
        "used_max_side": used_max_side,
        "error": error,
    }

    if status == "ok":
        done_ids.add(img_id)
    else:
        session_fail += 1

    session_done += 1
    save_results()

    if session_done == 1 or session_done % 25 == 0 or len(done_ids) == len(test_df):
        elapsed_all = time.time() - session_start
        avg = elapsed_all / max(session_done, 1)
        remain = len(test_df) - len(done_ids)
        print(
            f"[{len(done_ids):,}/{len(test_df):,}] "
            f"{len(done_ids)/len(test_df):.1%} | "
            f"{avg:.2f}s/개 | 경과 {hms(elapsed_all)} | "
            f"ETA {hms(avg*remain)} | fail={session_fail}",
            flush=True,
        )

final = pd.DataFrame(list(results.values())).sort_values("idx").reset_index(drop=True)
valid = (
    final["status"].eq("ok")
    & final["prediction"].astype(str).str.strip().str.lower().isin(["a","b","c","d"])
)

print("\nTOTAL:", len(final))
print("VALID:", int(valid.sum()))
print("ERROR:", int((~valid).sum()))

if len(final) != len(test_df) or not valid.all():
    bad = final.loc[~valid, ["idx","id","path","error"]]
    display(bad.head(20))
    raise RuntimeError(
        "미완료/오류 문항이 있습니다. 같은 셀을 다시 실행하면 정상 완료 ID는 건너뛰고 재시도합니다."
    )

pred_map = dict(zip(
    final["id"].astype(str),
    final["prediction"].astype(str).str.strip().str.lower()
))

submission = sample_submission[["id"]].copy()
submission["answer"] = submission["id"].astype(str).map(pred_map)

assert len(submission) == len(test_df)
assert submission["answer"].notna().all()
assert submission["answer"].isin(["a","b","c","d"]).all()

submission.to_csv(SUBMISSION_CSV, index=False, encoding="utf-8-sig")

print("\n" + "=" * 78)
print("SUBMISSION READY")
print("PATH:", SUBMISSION_CSV)
print("ROWS:", len(submission))
print("DISTRIBUTION:")
print(submission["answer"].value_counts().sort_index())
print("=" * 78)

display(submission.head(10))
